In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Input, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
import gc
import golois
import keras
planes = 31
moves = 361
N = 10000
batch = 128


input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')

In [7]:


# ——————————————————————————————————————————
# 🔧 1. Bloc Squeeze & Excitation
# ——————————————————————————————————————————

def se_block(input_tensor, filters, ratio=16):
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([input_tensor, se])

# ——————————————————————————————————————————
# 🔧 2. Bloc MobileNet avec SE intégré
# ——————————————————————————————————————————

def mobile_block_se(x, expand_filters, project_filters, stride=1, ratio=16, l2_reg=0.0001):
    # Expansion
    m = layers.Conv2D(expand_filters, kernel_size=1, padding='same',
                      use_bias=False, kernel_regularizer=regularizers.l2(l2_reg))(x)
    m = layers.BatchNormalization()(m)
    m = layers.Activation('relu')(m)

    # Depthwise Convolution (attention ici : depthwise_regularizer !)
    m = layers.DepthwiseConv2D(kernel_size=3, strides=stride, padding='same',
                               use_bias=False, depthwise_regularizer=regularizers.l2(l2_reg))(m)
    m = layers.BatchNormalization()(m)
    m = layers.Activation('relu')(m)

    # Projection
    m = layers.Conv2D(project_filters, kernel_size=1, padding='same',
                      use_bias=False, kernel_regularizer=regularizers.l2(l2_reg))(m)
    m = layers.BatchNormalization()(m)

    # Squeeze & Excitation
    m = se_block(m, project_filters, ratio)

    # Résidu (si même shape)
    if stride == 1 and x.shape[-1] == project_filters:
        m = layers.Add()([x, m])

    return m

# ——————————————————————————————————————————
# 🧠 3. Modèle SE-MobileNet
# ——————————————————————————————————————————

def build_se_mobilenet(input_shape=(19, 19, 21), blocks=32, expand=1152, project=192):
    inputs = Input(shape=input_shape)
    x = layers.Conv2D(project, 1, padding='same',
                      kernel_regularizer=regularizers.l2(0.0001))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for _ in range(blocks):
        x = mobile_block_se(x, expand_filters=expand, project_filters=project)

    # Policy head
    policy = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias=False)(x)
    policy = layers.Flatten()(policy)
    policy = layers.Activation('softmax', name='policy')(policy)

    # Value head
    value = layers.GlobalAveragePooling2D()(x)
    value = layers.Dense(50, activation='relu')(value)
    value = layers.Dense(1, activation='sigmoid', name='value')(value)

    return Model(inputs=inputs, outputs=[policy, value])

# ——————————————————————————————————————————
# ⚙️ 4. Entraînement du modèle
# ——————————————————————————————————————————

# Hyperparamètres
epochs = 250
batch_size = 32
policy_weight = 1.0
value_weight = 4.0

# Scheduler de learning rate (comme dans le papier)
def get_lr(epoch):
    if epoch < 100:
        return 0.0005
    elif epoch < 150:
        return 0.00005
    elif epoch < 200:
        return 0.000005
    else:
        return 0.0000005

# Création du modèle
model = build_se_mobilenet(blocks=32, expand=1152, project=192)
model.summary()

# Optimiseur
optimizer = Adam(learning_rate=get_lr(0))

# Compilation avec pertes pondérées
model.compile(optimizer=optimizer,
              loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
              loss_weights={'policy': policy_weight, 'value': value_weight},
              metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

# ——————————————————————————————————————————
# 🔁 5. Boucle d'entraînement
# ——————————————————————————————————————————

for epoch in range(1, epochs + 1):
    print(f"\n🔁 Epoch {epoch}")

    # MAJ du learning rate
    lr = get_lr(epoch)
    model.optimizer.learning_rate.assign(lr)
    print(f"→ Learning rate: {lr}")

    # Chargement dynamique du batch (adapté à ton pipeline Go)
    golois.getBatch(input_data, policy, value, end, groups, epoch * N)

    # Fit pour une époque
    model.fit(input_data,
              {'policy': policy, 'value': value},
              epochs=1,
              batch_size=batch_size,
              verbose=1)

    # Garbage collector pour libérer mémoire GPU
    if epoch % 5 == 0:
        gc.collect()

    # Évaluation et sauvegarde
    if epoch % epochs == 0:
        golois.getValidation(input_data, policy, value, end)
        val = model.evaluate(input_data, [policy, value], verbose=0, batch_size=batch_size)
        print(f"✅ Validation: {val}")
        model.save('best_se_mobilenet.h5')

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 19, 19,    │          0 │ -                 │
│ (InputLayer)        │ 21)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_200 (Conv2D) │ (None, 19, 19,    │      4,224 │ input_layer_4[0]… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │        768 │ conv2d_200[0][0]  │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 19, 19,    │          0 │ batch_normalizat… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_201 (Conv2D) │ (None, 19, 19,    │    221,184 │ re_lu_4[0][0]     │
│                     │ 1152)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │      4,608 │ conv2d_201[0][0]  │
│ (BatchNormalizatio… │ 1152)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_193      │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 1152)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_97 │ (None, 19, 19,    │     10,368 │ activation_193[0… │
│ (DepthwiseConv2D)   │ 1152)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │      4,608 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 1152)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_194      │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Activation)        │ 1152)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_202 (Conv2D) │ (None, 19, 19,    │    221,184 │ activation_194[0… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 19, 19,    │        768 │ conv2d_202[0][0]  │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 192)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_96          │ (None, 1, 1, 192) │          0 │ global_average_p… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_195 (Dense)   │ (None, 1, 1, 12)  │      2,304 │ reshape_96[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_196 (Dense)   │ (None, 1, 1, 192) │      2,304 │ dense_195[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_96         │ (None, 19, 19,    │          0 │ batch_normalizat… │
│ (Multiply)          │ 192)              │            │ dense_196[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 14,969,381 (57.10 MB)

 Trainable params: 14,809,253 (56.49 MB)

 Non-trainable params: 160,128 (625.50 KB)

: 